# OdinVTube — Personal Face Student (Colab)

Trains a **tiny low-RAM** model on **your face only**.

Code is cloned from:
**https://github.com/aquaticcalf/odin-vtube**

Pipeline:
1. Mount Drive (for your videos + outputs)
2. `git clone` the repo
3. Label videos with MediaPipe teacher
4. Train student
5. Export ONNX → Drive

**Runtime → Change runtime type → GPU** (T4 is fine).

## 0) Config — edit paths if needed

In [ ]:
# Repo (public or your private with Colab auth)
REPO_URL = "https://github.com/aquaticcalf/odin-vtube.git"
REPO_DIR = "/content/odin-vtube"
BRANCH = "main"  # or "master" if that is default

# Your data on Google Drive (same Google account)
VIDEO_GLOB = "/content/drive/MyDrive/odin_face/videos/*.mp4"
DATA_DIR   = "/content/drive/MyDrive/odin_face/dataset"
RUN_DIR    = "/content/drive/MyDrive/odin_face/runs/run1"

# Labeling
EVERY_N_FRAMES = 2
IMG_SIZE = 128
MAX_FRAMES = 0        # 0 = no limit; use 2000 for a smoke test

# Training
EPOCHS = 40
BATCH = 64
LR = 1e-3

CODE_ROOT = f"{REPO_DIR}/personal_face"
print("Config OK")

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

Path("/content/drive/MyDrive/odin_face/videos").mkdir(parents=True, exist_ok=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

print("Put your face MP4s here:")
print("  /content/drive/MyDrive/odin_face/videos/")

## 2) Clone odin-vtube from GitHub

In [ ]:
import os
import subprocess
from pathlib import Path

if Path(REPO_DIR, ".git").is_dir():
    print("Repo exists — pull latest")
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "--all"])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
else:
    # fresh clone
    if Path(REPO_DIR).exists():
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR])

assert Path(CODE_ROOT, "scripts", "label_teacher.py").is_file(), (
    f"Missing training scripts under {CODE_ROOT}. "
    "Check REPO_URL / BRANCH."
)
print("CODE_ROOT =", CODE_ROOT)
print("scripts:", sorted(p.name for p in Path(CODE_ROOT, "scripts").glob("*.py")))

## 3) Install dependencies

In [ ]:
# Colab already has torch / numpy. Install teacher + export deps.
# MediaPipe pulls opencv-contrib-python — do NOT also install opencv-python-headless
# (two OpenCV packages in one env causes cv2 conflicts).
# Latest PyPI (2026-07): mediapipe 0.10.35, onnx 1.22, onnxruntime 1.27, opencv 5.0.0.93
!pip -q install -U "mediapipe>=0.10.35" "onnx>=1.16" "onnxruntime>=1.17" tqdm Pillow
# Optional: force OpenCV 5 if Colab preinstall is older 4.x
!pip -q install -U "opencv-contrib-python>=5.0.0.93"

import cv2, mediapipe as mp, torch
print("cv2", cv2.__version__)
print("mediapipe", mp.__version__)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())


## 4) Find videos

In [ ]:
import glob

vids = sorted(glob.glob(VIDEO_GLOB))
print(f"found {len(vids)} videos:")
for v in vids:
    print(" -", v)
if not vids:
    raise FileNotFoundError(
        f"No videos at {VIDEO_GLOB}\n"
        "Upload MP4s to Drive/odin_face/videos/ then re-run this cell."
    )

## 5) Label videos (MediaPipe teacher)

In [ ]:
import subprocess
import shlex

cmd = [
    "python", f"{CODE_ROOT}/scripts/label_teacher.py",
    "--videos", *vids,
    "--out", DATA_DIR,
    "--img-size", str(IMG_SIZE),
    "--every-n", str(EVERY_N_FRAMES),
    "--max-frames", str(MAX_FRAMES),
]
print(" ".join(shlex.quote(c) for c in cmd))
subprocess.check_call(cmd)

In [ ]:
import json
from pathlib import Path

meta = json.loads(Path(DATA_DIR, "meta.json").read_text())
print(json.dumps(meta, indent=2))
print("previews:", len(list(Path(DATA_DIR, "preview").glob("*.jpg"))))

## 6) Train student

In [ ]:
import subprocess

cmd = [
    "python", f"{CODE_ROOT}/scripts/train_student.py",
    "--data", DATA_DIR,
    "--out", RUN_DIR,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--lr", str(LR),
    "--img-size", str(IMG_SIZE),
]
subprocess.check_call(cmd)

## 7) Export ONNX

In [ ]:
import subprocess
from pathlib import Path

onnx_path = str(Path(RUN_DIR) / "personal_face_student.onnx")
cmd = [
    "python", f"{CODE_ROOT}/scripts/export_onnx.py",
    "--ckpt", str(Path(RUN_DIR) / "best.pt"),
    "--out", onnx_path,
]
subprocess.check_call(cmd)

print("\n=== ARTIFACTS ON DRIVE ===")
print(onnx_path)
print(onnx_path.replace(".onnx", ".json"))
print(Path(RUN_DIR) / "best.pt")

## Done

On your PC, pull/download:
- `personal_face_student.onnx`
- `personal_face_student.json`

Then we wire them into OdinVTube for local tracking.